In [0]:
# PrepAmbiente

In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlsproyecto";

In [0]:
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para el almacenamiento administrado del catálogo';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS IN catalog_au;

###Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.taxi_trips
(
    id                      STRING,
    vendor_id               INT,
    pickup_datetime         TIMESTAMP,
    dropoff_datetime        TIMESTAMP,
    passenger_count         INT,
    pickup_longitude        DOUBLE,
    pickup_latitude         DOUBLE,
    dropoff_longitude       DOUBLE,
    dropoff_latitude        DOUBLE,
    store_and_fwd_flag      STRING,
    trip_duration           BIGINT,

    _ingestion_timestamp    TIMESTAMP,
    _source_file            STRING,
    _source_system          STRING,
    _batch_id               STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageName}.dfs.core.windows.net/taxi_trips/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.citibike_trips
(
    tripduration             BIGINT,
    starttime                TIMESTAMP,
    stoptime                 TIMESTAMP,

    start_station_id         INT,
    start_station_name       STRING,
    start_station_latitude   DOUBLE,
    start_station_longitude  DOUBLE,

    end_station_id           INT,
    end_station_name         STRING,
    end_station_latitude     DOUBLE,
    end_station_longitude    DOUBLE,

    bikeid                    INT,
    usertype                  STRING,
    birth_year                INT,
    gender                    INT,

    _ingestion_timestamp     TIMESTAMP,
    _source_file             STRING,
    _source_system           STRING,
    _batch_id                STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageName}.dfs.core.windows.net/citibike_trips/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.weather_hourly
(
    time                    TIMESTAMP,
    temperature_2m_c        DOUBLE,
    precipitation_mm        DOUBLE,
    rain_mm                 DOUBLE,

    cloudcover_pct          DOUBLE,
    cloudcover_low_pct      DOUBLE,
    cloudcover_mid_pct      DOUBLE,
    cloudcover_high_pct     DOUBLE,

    windspeed_10m_kmh       DOUBLE,
    winddirection_10m_deg   DOUBLE,

    _ingestion_timestamp    TIMESTAMP,
    _source_file            STRING,
    _source_system          STRING,
    _batch_id               STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageName}.dfs.core.windows.net/weather_hourly/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.taxi_zones_geojson
(
    objectid                INT,
    location_id             INT,
    zone                    STRING,
    borough                 STRING,
    shape_area              DOUBLE,
    shape_leng              DOUBLE,
    geometry                STRING,

    _ingestion_timestamp    TIMESTAMP,
    _source_file            STRING,
    _source_system          STRING,
    _batch_id               STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageName}.dfs.core.windows.net/taxi_zones_geojson/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.taxi_zones_csv
(
    objectid                INT,
    shape_leng              DOUBLE,
    the_geom                STRING,
    shape_area              DOUBLE,
    zone                    STRING,
    location_id             INT,
    borough                 STRING,

    _ingestion_timestamp    TIMESTAMP,
    _source_file            STRING,
    _source_system          STRING,
    _batch_id               STRING
)
USING DELTA
LOCATION 'abfss://bronze@${storageName}.dfs.core.windows.net/taxi_zones_csv/';

In [0]:
%sql
SHOW TABLES IN catalog_au.bronze;

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.taxi_trips
(
    id STRING,
    vendor_id INT,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    passenger_count INT,
    pickup_longitude DOUBLE,
    pickup_latitude DOUBLE,
    dropoff_longitude DOUBLE,
    dropoff_latitude DOUBLE,
    store_and_fwd_flag STRING,
    trip_duration BIGINT,

    _ingestion_timestamp TIMESTAMP,
    _source_file STRING,
    _source_system STRING,
    _batch_id STRING,

    dq_invalid_duration INT,
    dq_invalid_passenger_count INT,
    dq_invalid_pickup_coordinates INT,
    dq_invalid_dropoff_coordinates INT,
    dq_long_duration INT,
    dq_valid_trip INT,

    duration_minutes DOUBLE,
    duration_hours DOUBLE,
    pickup_date DATE,
    pickup_hour INT,
    pickup_day_of_week STRING,
    pickup_month INT,
    pickup_year INT,
    is_weekend INT
)
USING DELTA
LOCATION 'abfss://silver@${storageName}.dfs.core.windows.net/taxi_trips/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.citibike_trips
(
    trip_id STRING,

    tripduration BIGINT,
    starttime TIMESTAMP,
    stoptime TIMESTAMP,

    start_station_id INT,
    start_station_name STRING,
    start_station_latitude DOUBLE,
    start_station_longitude DOUBLE,

    end_station_id INT,
    end_station_name STRING,
    end_station_latitude DOUBLE,
    end_station_longitude DOUBLE,

    bikeid INT,
    usertype STRING,
    birth_year INT,
    gender INT,

    _ingestion_timestamp TIMESTAMP,
    _source_file STRING,
    _source_system STRING,
    _batch_id STRING,

    dq_invalid_duration INT,
    dq_duration_mismatch INT,
    dq_invalid_start_coordinates INT,
    dq_invalid_end_coordinates INT,
    dq_missing_birth_year INT,
    dq_age_outlier INT,
    dq_long_duration INT,
    dq_valid_trip INT,

    duration_minutes DOUBLE,
    duration_hours DOUBLE,

    start_date DATE,
    start_hour INT,
    start_day_of_week STRING,
    start_month INT,
    start_year INT,
    is_weekend INT,

    rider_age INT
)
USING DELTA
LOCATION 'abfss://silver@${storageName}.dfs.core.windows.net/citibike_trips/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.weather_hourly
(
    time TIMESTAMP,

    temperature_2m_c DOUBLE,
    precipitation_mm DOUBLE,
    rain_mm DOUBLE,

    cloudcover_pct DOUBLE,
    cloudcover_low_pct DOUBLE,
    cloudcover_mid_pct DOUBLE,
    cloudcover_high_pct DOUBLE,

    windspeed_10m_kmh DOUBLE,
    winddirection_10m_deg DOUBLE,

    _ingestion_timestamp TIMESTAMP,
    _source_file STRING,
    _source_system STRING,
    _batch_id STRING,

    dq_missing_core_weather INT,
    dq_missing_wind_direction INT,
    dq_invalid_precipitation INT,
    dq_invalid_rain INT,
    dq_invalid_cloudcover INT,
    dq_invalid_windspeed INT,
    dq_invalid_wind_direction INT,
    dq_valid_weather INT,

    weather_date DATE,
    weather_hour INT,
    weather_day_of_week STRING,
    weather_month INT,
    weather_year INT,
    is_weekend INT,

    has_precipitation INT,
    has_rain INT,
    weather_condition STRING
)
USING DELTA
LOCATION 'abfss://silver@${storageName}.dfs.core.windows.net/weather_hourly/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.taxi_zone_features
(
    objectid INT,
    location_id INT,
    zone STRING,
    borough STRING,

    shape_area DOUBLE,
    shape_leng DOUBLE,
    geometry STRING,

    dq_attribute_mismatch INT,

    _ingestion_timestamp TIMESTAMP,
    _source_file STRING,
    _source_system STRING,
    _batch_id STRING,
    _csv_source_file STRING,

    borough_category STRING,
    geometry_type STRING,

    dq_missing_geometry INT,
    dq_invalid_geometry_type INT,
    dq_valid_feature INT
)
USING DELTA
LOCATION 'abfss://silver@${storageName}.dfs.core.windows.net/taxi_zone_features/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.taxi_zones
(
    location_id INT,
    zone STRING,
    borough STRING,
    borough_category STRING,

    source_feature_count BIGINT,
    dq_valid_zone INT,

    _ingestion_timestamp TIMESTAMP,
    _source_file STRING,
    _source_system STRING,
    _batch_id STRING
)
USING DELTA
LOCATION 'abfss://silver@${storageName}.dfs.core.windows.net/taxi_zones/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.taxi_trips_enriched
(

    id STRING,
    vendor_id INT,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    passenger_count INT,

    pickup_longitude DOUBLE,
    pickup_latitude DOUBLE,
    dropoff_longitude DOUBLE,
    dropoff_latitude DOUBLE,

    store_and_fwd_flag STRING,
    trip_duration BIGINT,

    _ingestion_timestamp TIMESTAMP,
    _source_file STRING,
    _source_system STRING,
    _batch_id STRING,

    dq_invalid_duration INT,
    dq_invalid_passenger_count INT,
    dq_invalid_pickup_coordinates INT,
    dq_invalid_dropoff_coordinates INT,
    dq_long_duration INT,
    dq_valid_trip INT,

    duration_minutes DOUBLE,
    duration_hours DOUBLE,

    pickup_date DATE,
    pickup_hour INT,
    pickup_day_of_week STRING,
    pickup_month INT,
    pickup_year INT,
    is_weekend INT,

    pickup_zone_objectid INT,
    pickup_location_id INT,
    pickup_zone STRING,
    pickup_borough STRING,
    pickup_borough_category STRING,

    dropoff_zone_objectid INT,
    dropoff_location_id INT,
    dropoff_zone STRING,
    dropoff_borough STRING,
    dropoff_borough_category STRING,

    dq_missing_pickup_zone INT,
    dq_missing_dropoff_zone INT,
    dq_valid_spatial INT
)
USING DELTA
LOCATION 'abfss://silver@${storageName}.dfs.core.windows.net/taxi_trips_enriched/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.citibike_trips_enriched
(

    trip_id STRING,
    tripduration BIGINT,
    starttime TIMESTAMP,
    stoptime TIMESTAMP,

    start_station_id INT,
    start_station_name STRING,
    start_station_latitude DOUBLE,
    start_station_longitude DOUBLE,

    end_station_id INT,
    end_station_name STRING,
    end_station_latitude DOUBLE,
    end_station_longitude DOUBLE,

    bikeid INT,
    usertype STRING,
    birth_year INT,
    gender INT,

    -- ============================================================
    -- METADATA
    -- ============================================================

    _ingestion_timestamp TIMESTAMP,
    _source_file STRING,
    _source_system STRING,
    _batch_id STRING,


    dq_invalid_duration INT,
    dq_duration_mismatch INT,
    dq_invalid_start_coordinates INT,
    dq_invalid_end_coordinates INT,
    dq_missing_birth_year INT,
    dq_age_outlier INT,
    dq_long_duration INT,
    dq_valid_trip INT,

    duration_minutes DOUBLE,
    duration_hours DOUBLE,

    start_date DATE,
    start_hour INT,
    start_day_of_week STRING,
    start_month INT,
    start_year INT,
    is_weekend INT,

    rider_age INT,

    start_zone_objectid INT,
    start_location_id INT,
    start_zone STRING,
    start_borough STRING,
    start_borough_category STRING,

    end_zone_objectid INT,
    end_location_id INT,
    end_zone STRING,
    end_borough STRING,
    end_borough_category STRING,

    dq_missing_start_zone INT,
    dq_missing_end_zone INT,
    dq_valid_spatial INT
)
USING DELTA
LOCATION 'abfss://silver@${storageName}.dfs.core.windows.net/citibike_trips_enriched/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.mobility_events
(

    event_id STRING,
    source_event_id STRING,
    transport_type STRING,

    start_datetime TIMESTAMP,
    end_datetime TIMESTAMP,

    duration_seconds BIGINT,
    duration_minutes DOUBLE,
    duration_hours DOUBLE,


    start_latitude DOUBLE,
    start_longitude DOUBLE,
    end_latitude DOUBLE,
    end_longitude DOUBLE,


    start_location_id INT,
    start_zone STRING,
    start_borough STRING,
    start_borough_category STRING,


    end_location_id INT,
    end_zone STRING,
    end_borough STRING,
    end_borough_category STRING,


    event_date DATE,
    event_hour_ts TIMESTAMP,
    event_hour INT,
    event_day_of_week STRING,
    event_month INT,
    event_year INT,
    is_weekend INT,


    temperature_2m_c DOUBLE,
    precipitation_mm DOUBLE,
    rain_mm DOUBLE,
    cloudcover_pct DOUBLE,
    windspeed_10m_kmh DOUBLE,
    winddirection_10m_deg DOUBLE,
    weather_condition STRING,
    has_precipitation INT,
    has_rain INT,

    dq_invalid_duration INT,
    dq_invalid_start_coordinates INT,
    dq_invalid_end_coordinates INT,
    dq_valid_spatial INT,
    dq_missing_weather INT,
    dq_valid_event INT,


    passenger_count INT,
    usertype STRING,
    rider_age INT,


    _ingestion_timestamp TIMESTAMP,
    _source_file STRING,
    _source_system STRING,
    _batch_id STRING
)
USING DELTA
LOCATION 'abfss://silver@${storageName}.dfs.core.windows.net/mobility_events/';

###Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.mobility_hourly
(
    event_date DATE,
    event_hour INT,
    event_day_of_week STRING,
    event_month INT,
    is_weekend INT,

    start_location_id INT,
    start_zone STRING,
    start_borough STRING,
    start_borough_category STRING,

    transport_type STRING,

    total_trips BIGINT,
    avg_duration_minutes DOUBLE,
    min_duration_minutes DOUBLE,
    max_duration_minutes DOUBLE,

    _load_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://golden@${storageName}.dfs.core.windows.net/mobility_hourly/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.mobility_weather
(
    event_date DATE,
    event_hour INT,

    transport_type STRING,

    weather_condition STRING,
    has_precipitation INT,
    has_rain INT,

    total_trips BIGINT,
    avg_duration_minutes DOUBLE,

    temperature_2m_c DOUBLE,
    precipitation_mm DOUBLE,
    rain_mm DOUBLE,
    cloudcover_pct DOUBLE,
    windspeed_10m_kmh DOUBLE,
    winddirection_10m_deg DOUBLE,

    _load_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://golden@${storageName}.dfs.core.windows.net/mobility_weather/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.mobility_zone
(
    start_location_id INT,
    start_zone STRING,
    start_borough STRING,
    start_borough_category STRING,

    transport_type STRING,

    total_trips BIGINT,
    active_days BIGINT,

    avg_duration_minutes DOUBLE,

    weekend_trips BIGINT,
    rainy_trips BIGINT,

    peak_hour INT,

    avg_trips_per_day DOUBLE,

    _load_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://golden@${storageName}.dfs.core.windows.net/mobility_zone/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.transport_comparison
(
    event_date DATE,
    event_hour INT,
    event_day_of_week STRING,
    event_month INT,
    is_weekend INT,

    start_location_id INT,
    start_zone STRING,
    start_borough STRING,
    start_borough_category STRING,

    total_trips BIGINT,

    taxi_trips BIGINT,
    bike_trips BIGINT,

    taxi_share_pct DOUBLE,
    bike_share_pct DOUBLE,

    avg_taxi_duration_minutes DOUBLE,
    avg_bike_duration_minutes DOUBLE,

    temperature_2m_c DOUBLE,
    precipitation_mm DOUBLE,
    rain_mm DOUBLE,
    weather_condition STRING,

    _load_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://golden@${storageName}.dfs.core.windows.net/transport_comparison/';